In [2]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00


In [3]:
# connect colab with drive
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pickle
import sklearn

from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
cd /content/drive/My Drive/YoloFlyCam_Train

/content/drive/My Drive/YoloFlyCam_Train


In [5]:
!yolo version


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.3.226


In [6]:
!cat data.yaml

# data.yaml – cấu hình dataset cho YOLOv8
path: /content/drive/MyDrive/YoloFlyCam_Train  # đường dẫn gốc đến dataset
train: images/train
val: images/val
test: images/test

# Thư mục labels mới (tự động nhận nếu cùng cấp Images, nhưng có thể chỉ rõ nếu khác)
names:
  0: Human

# Nếu muốn chỉ rõ đường dẫn labels mới (tùy chọn)
train_label: Labels_new/train
val_label: Labels_new/val
test_label: Labels_new/test


In [19]:
# save as create_labels_from_yolo.py
import os
from pathlib import Path
from ultralytics import YOLO
from PIL import Image

# ---------- CONFIG ----------
MODEL_WEIGHTS = 'best.pt'   # hoặc 'yolov8n.pt'
CONF_THRESH = 0.30             # ngưỡng confidence để giữ detection
IOU_THRESH = 0.45              # NMS IoU (ultralytics xử lý NMS nội bộ)
class_name_to_keep = 'Human'  # lớp bạn muốn giữ (thường 'person')
# Thư mục ảnh (thay theo thực tế)
BASE = Path('/content/drive/MyDrive/YoloFlyCam_Train')  # ví dụ Colab + Drive
IMAGE_SUBFOLDERS = {
    'train': BASE / 'images' / 'train',
    'val':   BASE / 'images' / 'val',
    'test':  BASE / 'images' / 'test'
}
# Nơi lưu nhãn mới (sẽ tạo song song với Images)
OUT_LABEL_ROOT = BASE / 'Labels_new'   # sẽ tạo các thư mục train/val/test dưới đây
BACKUP_EXISTING_LABELS = True          # nếu True, sẽ backup labels cũ (nếu có)
# ----------------------------

# Helper: convert xyxy -> normalized yolo xywh
def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    # normalize
    return x_center / img_w, y_center / img_h, w / img_w, h / img_h

def ensure_dir(p: Path):
    if not p.exists():
        p.mkdir(parents=True, exist_ok=True)

def backup_labels_folder(src_labels_folder: Path):
    if not src_labels_folder.exists():
        return None
    backup_folder = src_labels_folder.with_name(src_labels_folder.name + '_backup')
    if backup_folder.exists():
        # create numbered backup if already exists
        i = 1
        while backup_folder.with_name(backup_folder.name + f'_{i}').exists():
            i += 1
        backup_folder = backup_folder.with_name(backup_folder.name + f'_{i}')
    src_labels_folder.rename(backup_folder)
    return backup_folder

# Load model
print("Loading model:", MODEL_WEIGHTS)
model = YOLO(MODEL_WEIGHTS)  # model will auto-download if needed

# find index of person class in model.names (safety)
person_class_indices = [i for i, n in model.names.items() if 'person' in str(n).lower()]
if len(person_class_indices) == 0:
    print("Warning: model has no 'person' class in model.names:", model.names)
    # we'll try to map class 0 by default (common for COCO)
    person_class_indices = [0]
else:
    print("Person class indices in model:", person_class_indices)

# Process each subfolder
for split, img_folder in IMAGE_SUBFOLDERS.items():
    print(f"\nProcessing split '{split}' images in:", img_folder)
    if not img_folder.exists():
        print("  -> Folder not found, skipping.")
        continue

    out_label_folder = OUT_LABEL_ROOT / split
    ensure_dir(out_label_folder)

    # optionally backup existing labels in same-level 'Labels' folder
    original_labels_folder = img_folder.parent.parent / 'Labels' / split  # guess typical structure Images/train -> Labels/train
    if BACKUP_EXISTING_LABELS and original_labels_folder.exists():
        print(f"Backing up existing labels folder {original_labels_folder}")
        try:
            backup = backup_labels_folder(original_labels_folder)
            print("  backed up to:", backup)
        except Exception as e:
            print("  backup failed:", e)

    # iterate images
    image_paths = sorted([p for p in img_folder.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    print(f"  Found {len(image_paths)} images.")
    if len(image_paths) == 0:
        continue

    for img_path in image_paths:
        try:
            results = model.predict(source=str(img_path), conf=CONF_THRESH, imgsz=640, verbose=False)
            # results is a list; take first
            r = results[0]
            # image size
            pil = Image.open(img_path)
            img_w, img_h = pil.size
            pil.close()

            # get boxes
            boxes = r.boxes  # Boxes object
            # boxes.xyxy, boxes.conf, boxes.cls available
            xyxy_list = boxes.xyxy.tolist() if boxes is not None and len(boxes) > 0 else []
            conf_list = boxes.conf.tolist() if boxes is not None and len(boxes) > 0 else []
            cls_list = boxes.cls.tolist() if boxes is not None and len(boxes) > 0 else []

            # keep only person detections and above conf threshold (model already filtered by conf)
            yolo_lines = []
            for xyxy, conf, cls in zip(xyxy_list, conf_list, cls_list):
                cls = int(cls)
                if cls not in person_class_indices:
                    continue
                x1, y1, x2, y2 = xyxy
                x_c, y_c, w, h = xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)
                # clamp values 0..1
                x_c = min(max(x_c, 0.0), 1.0)
                y_c = min(max(y_c, 0.0), 1.0)
                w = min(max(w, 0.0), 1.0)
                h = min(max(h, 0.0), 1.0)
                # class id in new labels: we will use 0 for 'person' (user must ensure data.yaml names align)
                yolo_lines.append(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

            # write label file (even if empty, we may skip writing empty)
            out_label_path = out_label_folder / (img_path.stem + '.txt')
            if len(yolo_lines) == 0:
                # Write empty file (or choose to skip creating file)
                # If you prefer not to create empty label files, comment next two lines
                with open(out_label_path, 'w') as f:
                    f.write('')
            else:
                with open(out_label_path, 'w') as f:
                    f.write('\n'.join(yolo_lines))

        except Exception as e:
            print("  Error on", img_path.name, "-", e)

    print(f"  Saved labels to {out_label_folder}")

print("\nDone. New labels saved under:", OUT_LABEL_ROOT)
print("Remember to update your data.yaml to point to new image/label folders if you want to train with these labels.")


Loading model: best.pt

Processing split 'train' images in: /content/drive/MyDrive/YoloFlyCam_Train/images/train
  Found 795 images.
  Saved labels to /content/drive/MyDrive/YoloFlyCam_Train/Labels_new/train

Processing split 'val' images in: /content/drive/MyDrive/YoloFlyCam_Train/images/val
  Found 292 images.
  Saved labels to /content/drive/MyDrive/YoloFlyCam_Train/Labels_new/val

Processing split 'test' images in: /content/drive/MyDrive/YoloFlyCam_Train/images/test
  Found 67 images.
  Saved labels to /content/drive/MyDrive/YoloFlyCam_Train/Labels_new/test

Done. New labels saved under: /content/drive/MyDrive/YoloFlyCam_Train/Labels_new
Remember to update your data.yaml to point to new image/label folders if you want to train with these labels.


In [7]:
from ultralytics import YOLO
model = YOLO('best.pt')

model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='disaster_aug',
    augment=True,   # bật augmentation (mặc định True)
    mosaic=1.0,     # mosaic intensity
    mixup=0.2,      # mixup probability
    device=0
)


Ultralytics 8.3.226 🚀 Python-3.12.12 torch-2.8.0+cu126 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [62]:
model = YOLO('best.pt')
metrics = model.val(data='data.yaml')
print(metrics)

Ultralytics 8.3.226 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.6 ms, read: 178.3±150.3 MB/s, size: 3164.7 KB)
val: Scanning /content/drive/MyDrive/YoloFlyCam_Train/labels/val... 292 images, 155 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 31.8it/s 9.2s
val: New cache created: /content/drive/MyDrive/YoloFlyCam_Train/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 0.2it/s 1:20
                   all        292        165      0.831      0.747      0.868      0.749
Speed: 5.3ms preprocess, 169.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/drive/MyDrive/YoloFlyCam_Train/runs/detect/val2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_ma

In [30]:
from IPython.display import Image, display

# Xem kết quả tổng hợp
display(Image(filename='/content/drive/My Drive/YoloFlyCam_Train/runs/detect/val/BoxF1_.png'))

# (Tùy chọn) xem ma trận nhầm lẫn
display(Image(filename='/content/drive/My Drive/YoloFlyCam_Train/runs/detect/trvalain/confusion_matrix.png'))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/YoloFlyCam_Train/runs/detect/train/results.png'